# ATMOS — Analytical Methods

This notebook documents every formula used in `src/atmos.py`, rendered as equations,
with worked examples that call the production functions directly.

**Scope:** atmospheric property interpolation, compressible dynamic pressure,
speed-of-sound, and all four airspeed conversions (subsonic *and* supersonic).

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '20_ATMOS'))
# If running from tools/, the project root is one level up
sys.path.insert(0, os.path.abspath('..'))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from src.atmos import (
    set_atmos_model,
    get_atmos_prop_alt,
    get_atmos_prop_pres,
    mach_alt, tas_alt, eas_alt, cas_alt,
    GAMMA, A_0_KTS, P_0_PSF,
)

# Point to the atmospheric model CSV (relative to project root)
set_atmos_model('../data/models/standard_atmos.csv')

print('Imports OK')
print(f'  γ  = {GAMMA}')
print(f'  a₀ = {A_0_KTS} kts')
print(f'  p₀ = {P_0_PSF} psf')

---
## 1  Physical constants

| Symbol | Value | Meaning |
|--------|-------|---------|
| $\gamma$ | 1.4 | Ratio of specific heats for dry air |
| $a_0$ | 661.4745 kts | Sea-level speed of sound (ISA) |
| $p_0$ | 2116.224 psf | Sea-level static pressure (ISA) |

These are thermodynamic facts, not tunable parameters.

---
## 2  Atmospheric model — table interpolation

The module does **not** solve the hydrostatic equations at runtime.
Instead it reads a pre-computed CSV (e.g. `data/models/standard_atmos.csv`) and
uses `numpy.interp` for linear interpolation between table rows.

Given pressure altitude $H$ (ft), the lookup returns:

| Output | Column | Symbol |
|--------|--------|--------|
| Static pressure | `P_lb/ft2` | $p$ (psf) |
| Density ratio | `pho/pho0` | $\sigma = \rho/\rho_0$ |
| Air density | `pho_slug/ft3` | $\rho$ (slug/ft³) |
| Temperature | `T_degR` | $T$ (°R) |

In [ ]:
# Example: atmospheric properties at three altitudes
altitudes_ft = [0, 10_000, 20_000, 35_000, 50_000]

rows = []
for h in altitudes_ft:
    a = get_atmos_prop_alt(h)
    rows.append({
        'H (ft)':      h,
        'p (psf)':     round(a['p_static_psf'], 2),
        'σ = ρ/ρ₀':   round(a['pho_ratio'], 4),
        'ρ (slug/ft³)': f"{a['pho_slug_ft3']:.6f}",
        'T (°R)':      round(a['temp_degR'], 2),
    })

pd.DataFrame(rows).set_index('H (ft)')

---
## 3  Local speed of sound

For a perfect gas the speed of sound is

$$a = \sqrt{\frac{\gamma\, p}{\rho}}$$

where $p$ is static pressure (psf) and $\rho$ is density (slug/ft³).
The raw result is in ft/s; multiplying by the factor $k = 0.5924838$
converts to knots:

$$a_{\text{kts}} = k\sqrt{\frac{\gamma\, p}{\rho}}$$

At sea level this gives exactly $a_0 = 661.4745$ kts.
The sea-level speed of sound is only used as a reference; the *local* value
varies with altitude via the change in $p$ and $\rho$.

In [ ]:
K_FTS_TO_KTS = 0.5924838  # sqrt(psf / (slug/ft³))  →  knots

def local_speed_of_sound(h_ft):
    a = get_atmos_prop_alt(h_ft)
    return math.sqrt(GAMMA * a['p_static_psf'] / a['pho_slug_ft3']) * K_FTS_TO_KTS

alts = range(0, 55_000, 5_000)
a_kts = [local_speed_of_sound(h) for h in alts]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(a_kts, alts, 'b-o', markersize=4)
ax.set_xlabel('Speed of sound  a  (kts)')
ax.set_ylabel('Pressure altitude  H  (ft)')
ax.set_title('Local speed of sound vs altitude (ISA)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f'Sea-level check: a₀ = {local_speed_of_sound(0):.4f} kts  (target {A_0_KTS} kts)')

---
## 4  Compressible impact pressure  $q_c$

Impact (compressible dynamic) pressure $q_c$ is the difference between
stagnation and static pressure.  It depends on Mach number and local
static pressure.

### 4.1  Subsonic  ($M \le 1$)

Derived from isentropic relations:

$$q_c = p\left[\left(1 + \frac{\gamma-1}{2}M^2\right)^{\gamma/(\gamma-1)} - 1\right]$$

With $\gamma = 1.4$ this simplifies to

$$q_c = p\left[\left(1 + 0.2\,M^2\right)^{7/2} - 1\right]$$

### 4.2  Supersonic  ($M > 1$) — Rayleigh pitot formula

A normal shock forms ahead of the pitot tube, so the stagnation pressure
is reduced.  The Rayleigh formula is:

$$q_c = p\left[\frac{(\gamma+1)}{2}M^2 \cdot\left(\frac{(1+\gamma)^2 M^2}{4\gamma M^2 - 2(\gamma-1)}\right)^{1/(\gamma-1)} - 1\right]$$

With $\gamma = 1.4$:

$$q_c = p\left[\frac{6}{5}M^2 \cdot\left(\frac{36\,M^2}{7\,M^2 - 1}\right)^{5/2} - 1\right]$$

In [ ]:
def qc(M, p):
    """Compressible dynamic pressure — mirrors _dynamic_pressure() in atmos.py."""
    if M <= 1.0:
        return p * ((1 + 0.2 * M**2)**(7/2) - 1)
    q_a = (GAMMA + 1) / 2 * M**2
    q_b = (((1 + GAMMA)**2 * M**2) / (4*GAMMA*M**2 - 2*(GAMMA - 1)))**(1/(GAMMA - 1))
    return (q_a * q_b - 1) * p

p_sl = P_0_PSF  # sea level, for a clean reference curve
mach_range = np.linspace(0.2, 2.0, 200)
qc_values  = [qc(M, p_sl) for M in mach_range]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mach_range, qc_values, 'k-', linewidth=1.5)
ax.axvline(1.0, color='r', linestyle='--', linewidth=0.8, label='M = 1')
ax.set_xlabel('Mach number  M')
ax.set_ylabel('$q_c$  (psf)')
ax.set_title('Impact pressure $q_c$ vs Mach (p = p₀ = 2116.2 psf)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Show the discontinuity in the derivative at M=1 (formula switches)
print(f'q_c at M=0.99  (subsonic formula):    {qc(0.99,  p_sl):.2f} psf')
print(f'q_c at M=1.00  (subsonic formula):    {qc(1.00,  p_sl):.2f} psf')
print(f'q_c at M=1.01  (supersonic formula):  {qc(1.01,  p_sl):.2f} psf')

---
## 5  Calibrated airspeed  $V_{\text{CAS}}$

$V_{\text{CAS}}$ is defined as the airspeed an aircraft would have at sea level
if the measured impact pressure $q_c$ were generated under ISA sea-level conditions.
This makes CAS independent of altitude.

### 5.1  Subsonic ($V_{\text{CAS}} \le a_0$)

Inverting the subsonic $q_c$ relation at sea level:

$$V_{\text{CAS}} = a_0\sqrt{5\left[\left(\frac{q_c}{p_0}+1\right)^{2/7}-1\right]}$$

### 5.2  Supersonic ($V_{\text{CAS}} > a_0$)

The supersonic VCAS equation cannot be inverted analytically.  The code
uses the function `vcas_super(q_c)` which iterates on $V_{\text{CAS}}$
until the Rayleigh formula evaluated at $V_{\text{CAS}}/a_0$ matches $q_c$
to within a configurable tolerance.

Algorithm (bisection-style):
1. Start at $V_{\text{CAS}} = a_0$.
2. Compute $q_c^{\text{guess}}$ using the supersonic formula with $M_{\text{ref}} = V_{\text{CAS}}/a_0$.
3. If $q_c^{\text{guess}} < q_c$: increase $V_{\text{CAS}}$ by $\delta$.
4. If $q_c^{\text{guess}} > q_c$: halve $\delta$, then decrease $V_{\text{CAS}}$.
5. Repeat until $|q_c - q_c^{\text{guess}}| < \varepsilon \cdot q_c$.

In [ ]:
def kcas_from_qc(q_c):
    """KCAS from impact pressure — mirrors _kcas_from_qc() in atmos.py."""
    from src.atmos import vcas_super
    kcas = A_0_KTS * math.sqrt(5 * ((q_c / P_0_PSF + 1)**(2/7) - 1))
    if kcas > A_0_KTS:
        kcas = vcas_super(q_c)
    return kcas

# KCAS as a function of Mach at sea level (p = p₀)
cas_values = [kcas_from_qc(qc(M, P_0_PSF)) for M in mach_range]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mach_range, cas_values, 'b-', linewidth=1.5)
ax.axvline(1.0, color='r', linestyle='--', linewidth=0.8, label='M = 1')
ax.axhline(A_0_KTS, color='g', linestyle=':', linewidth=0.8, label=f'$a_0$ = {A_0_KTS} kts')
ax.set_xlabel('Mach number  M')
ax.set_ylabel('$V_{CAS}$  (kts)')
ax.set_title('CAS vs Mach at sea level (p = p₀)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 6  Airspeed relationships

Four airspeeds are computed for every condition.  The chain of
dependencies is shown below.

### 6.1  True airspeed  $V_{\text{TAS}}$

$$V_{\text{TAS}} = M \cdot a$$

where $a$ is the local speed of sound at the flight condition.

### 6.2  Equivalent airspeed  $V_{\text{EAS}}$

EAS normalises TAS for the change in air density:

$$V_{\text{EAS}} = V_{\text{TAS}}\sqrt{\sigma}$$

where the density ratio $\sigma = \rho/\rho_0$.

### 6.3  Forward conversions from each input type

| Given | Intermediate steps |
|-------|-------------------|
| $M$, $H$ | $a \to V_{\text{TAS}} = M a \to V_{\text{EAS}} = V_{\text{TAS}}\sqrt{\sigma} \to q_c \to V_{\text{CAS}}$ |
| $V_{\text{TAS}}$, $H$ | $M = V_{\text{TAS}}/a \to$ same chain |
| $V_{\text{EAS}}$, $H$ | $V_{\text{TAS}} = V_{\text{EAS}}/\sqrt{\sigma} \to M \to$ same chain |
| $V_{\text{CAS}}$, $H$ | $q_c$ from CAS definition $\to M = f(q_c, p) \to V_{\text{TAS}} \to V_{\text{EAS}}$ |

For the KCAS→Mach step the subsonic inversion is:

$$M = \sqrt{5\left[\left(\frac{q_c}{p}+1\right)^{2/7}-1\right]}$$

If $M > 1$ this is solved iteratively by `mach_super(q_c, p)`.

In [ ]:
# Visualise the four airspeeds along a constant-Mach cruise
M_cruise = 0.84
alts_ft = range(0, 52_000, 1_000)

speeds = [mach_alt(M_cruise, h) for h in alts_ft]
ktas_vals = [s['ktas'] for s in speeds]
keas_vals = [s['keas'] for s in speeds]
kcas_vals = [s['kcas'] for s in speeds]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ktas_vals, alts_ft, label='$V_{TAS}$', linewidth=1.5)
ax.plot(keas_vals, alts_ft, label='$V_{EAS}$', linewidth=1.5)
ax.plot(kcas_vals, alts_ft, label='$V_{CAS}$', linewidth=1.5)
ax.axvline(M_cruise * A_0_KTS, color='k', linestyle=':', linewidth=0.8,
           label=f'$M a_0$ = {M_cruise*A_0_KTS:.1f} kts  (TAS if $a=a_0$)')
ax.set_xlabel('Airspeed (kts)')
ax.set_ylabel('Pressure altitude  H  (ft)')
ax.set_title(f'Airspeed comparison — constant Mach {M_cruise} (ISA)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 7  Supersonic iterative solvers

Two functions use the same bisection-style algorithm:

| Function | Solves for | Converges when |
|----------|-----------|----------------|
| `vcas_super(q_c)` | $V_{\text{CAS}}$ | $\lvert q_c - q_c^{\text{guess}}(V_{\text{CAS}}) \rvert < \varepsilon \cdot q_c$ |
| `mach_super(q_c, p)` | $M$ | $\lvert q_c - q_c^{\text{guess}}(M, p) \rvert < \varepsilon \cdot q_c$ |

Solver tolerances ($\varepsilon$, initial step $\delta$) are read from
`config/defaults.json` at runtime — they are *not* hard-coded constants.

### Convergence sketch

Each iteration the step $\delta$ is halved when the solver overshoots,
producing a sequence of brackets that halve in width — equivalent to a
bisection search on an implicit function.

In [ ]:
# Demonstrate vcas_super convergence on a single q_c value
from src.atmos import vcas_super, mach_super, _dynamic_pressure
from src.config import APP_CONFIG

# Pick M=1.5 at 35,000 ft as a supersonic reference
h_test_ft = 35_000
M_test    = 1.5

atmos_test = get_atmos_prop_alt(h_test_ft)
p_test     = atmos_test['p_static_psf']
q_c_test   = _dynamic_pressure(M_test, p_test)

V_CAS_result = vcas_super(q_c_test)
M_result     = mach_super(q_c_test, p_test)

print(f'Input:          M = {M_test},  H = {h_test_ft:,} ft')
print(f'  p_static    = {p_test:.3f} psf')
print(f'  q_c         = {q_c_test:.3f} psf')
print(f'  VCAS result = {V_CAS_result:.4f} kts')
print(f'  Mach result = {M_result:.6f}  (should recover {M_test})')

---
## 8  Worked examples

### 8.1  Subsonic cruise — Mach 0.84 at 35,000 ft

In [ ]:
result = mach_alt(0.84, 35_000)

print('Input:  M = 0.84,  H = 35,000 ft')
print(f"  Mach  = {result['Mach']:.4f}")
print(f"  KTAS  = {result['ktas']:.2f} kts")
print(f"  KEAS  = {result['keas']:.2f} kts")
print(f"  KCAS  = {result['kcas']:.2f} kts")
print(f"  q_c   = {result['q_c']:.3f} psf")

### 8.2  Round-trip consistency — CAS → back to Mach

Start with Mach + altitude, extract the KCAS, then convert KCAS + altitude
back and verify Mach is recovered.

In [ ]:
for M_in, h_in in [(0.30, 5_000), (0.70, 25_000), (0.84, 35_000), (1.20, 40_000), (1.50, 45_000)]:
    fwd  = mach_alt(M_in, h_in)
    back = cas_alt(fwd['kcas'], h_in)
    err  = abs(back['Mach'] - M_in)
    ok   = '✓' if err < 1e-4 else f'ERR {err:.2e}'
    print(f'M={M_in:.2f}  H={h_in:>6,} ft  →  KCAS={fwd["kcas"]:6.2f}  →  M_recovered={back["Mach"]:.6f}  {ok}')

### 8.3  All four input types at the same flight condition

The four public functions (`mach_alt`, `tas_alt`, `eas_alt`, `cas_alt`) should
return identical output when given equivalent inputs.

In [ ]:
H_REF = 30_000  # ft
M_REF = 0.75

ref   = mach_alt(M_REF, H_REF)
by_m  = mach_alt(ref['Mach'], H_REF)
by_t  = tas_alt(ref['ktas'],  H_REF)
by_e  = eas_alt(ref['keas'],  H_REF)
by_c  = cas_alt(ref['kcas'],  H_REF)

cols = ['Mach', 'ktas', 'keas', 'kcas', 'q_c']
rows = {}
for label, r in [('mach_alt', by_m), ('tas_alt', by_t),
                  ('eas_alt', by_e), ('cas_alt', by_c)]:
    rows[label] = {k: round(r[k], 4) for k in cols}

pd.DataFrame(rows).T

### 8.4  Lookup by pressure (reverse interpolation)

`get_atmos_prop_pres` reverses the table: given a measured static pressure,
it returns the equivalent pressure altitude and other properties.

In [ ]:
# Round-trip: altitude → pressure → altitude
test_alts = [5_000, 18_000, 35_000, 50_000]

print(f'{"H_in (ft)":>12}  {"p (psf)":>12}  {"H_recovered (ft)":>20}  {"error (ft)":>12}')
for h in test_alts:
    p = get_atmos_prop_alt(h)['p_static_psf']
    h_back = get_atmos_prop_pres(p)['h_press_ft']
    print(f'{h:>12,}  {p:>12.3f}  {h_back:>20.4f}  {abs(h_back-h):>12.6f}')

---
## 9  Summary of formula chain

```
Inputs: { M or V_TAS or V_EAS or V_CAS }  +  H (ft)
         │
         ▼
  Table lookup / interpolation
    p(H),  ρ(H),  σ(H) = ρ/ρ₀
         │
         ▼
  Local speed of sound
    a = k · √(γ p / ρ)
         │
         ▼  (entry point determines direction)
  Mach  ←→  V_TAS = M · a
                  ↕
             V_EAS = V_TAS · √σ
                  ↕
             q_c  = f(M, p)    ← subsonic or Rayleigh (supersonic)
                  ↕
             V_CAS = g(q_c)    ← subsonic closed-form or vcas_super() iteration
```

All five quantities `{Mach, V_TAS, V_EAS, V_CAS, q_c}` are returned regardless
of which was the input.